<a href="https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis (The Grain):

One row represents one unique content item (content_hash_id) over a specific historical performance snapshot.

Time Window:

We will use a mid-panel month (March 2026) as our observation window to build features, predicting the decay proxy (Label) in the subsequent 90 days. We deliberately avoid June 2026 as it is our blind test set.



In [ ]:
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("1. Loading dim_content directly...")
df_dim = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    storage_options={"token": hf_token}
)

print("2. Loading ONLY March 2026 partition (Zero RAM overhead)...")
df_fact_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/",
    storage_options={"token": hf_token}
)

print("3. Aggregating to our target Grain (1 Row = 1 Content URL)...")
march_agg = df_fact_march.groupby('content_hash_id').agg({
    'gsc_clicks': 'sum',
    'gsc_impressions': 'sum',
    'gsc_avg_position': 'mean'
}).reset_index()

march_agg = march_agg.rename(columns={
    'gsc_clicks': 'clicks_past_30d',
    'gsc_impressions': 'impressions_past_30d',
    'gsc_avg_position': 'avg_position_past_30d'
})

print("4. Merging and finalizing Data Contract features...")
df = pd.merge(df_dim, march_agg, on='content_hash_id', how='inner')

df['target_proxy_decay'] = np.where(df['impressions_past_30d'] < df['impressions_past_30d'].median(), 1, 0)

print(f"\n--- SUCCESS ---")
print(f"Total Rows: {len(df)}")
print(f"Is one row = one unique content_hash_id? {df['content_hash_id'].is_unique}")

1. Loading dim_content directly...
2. Loading ONLY March 2026 partition (Zero RAM overhead)...
3. Aggregating to our target Grain (1 Row = 1 Content URL)...
4. Merging and finalizing Data Contract features...

--- SUCCESS ---
Total Rows: 331437
Is one row = one unique content_hash_id? True


## 2. Fields: feature / label / context / excluded

Features (Knowable at decision time):

impressions_past_30d, clicks_past_30d, avg_position_past_30d (Aggregated historical performance from Google Search Console).

Label (Proxy for prediction):

target_proxy_decay (1 if trend indicates decay, 0 otherwise). Used only during training to teach the model what decay looks like.

Context:

content_hash_id (Used for row identification and joins, must never be passed to the model).

Excluded:

Any future metrics (e.g., clicks in April). Excluded explicitly because it violates causality (Data Leakage). We cannot know future clicks when deciding to refresh a page in March.

In [ ]:
features = ['impressions_past_30d', 'clicks_past_30d', 'avg_position_past_30d']
label = 'target_proxy_decay'
context = 'content_hash_id'
excluded = 'future_clicks'

print(f"Features (Knowable): {features}")
print(f"Label (Prediction Proxy): {label}")
print(f"Context (Must not train on): {context}")
print(f"Excluded (To prevent leakage): {excluded}")

Features (Knowable): ['impressions_past_30d', 'clicks_past_30d', 'avg_position_past_30d']
Label (Prediction Proxy): target_proxy_decay
Context (Must not train on): content_hash_id
Excluded (To prevent leakage): future_clicks


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we prove our contract claims using queries:

Verifying the grain uniqueness.

Checking the availability of our features (missing values).

The Trap: Demonstrating that if we accidentally leak the label into the features, the data is invalid for decision-support.


In [ ]:
print(f"Row count ({df.shape[0]}) matches unique IDs ({df['content_hash_id'].nunique()}).")

print("\nMissing values in features:")
print(df[features].isnull().sum())

df['LEAKED_future_data'] = df['target_proxy_decay']
print("\nFact 3: THE TRAP")
print("If we train using 'LEAKED_future_data', the model will cheat with 100% accuracy.")
df = df.drop(columns=['LEAKED_future_data'])
print("Trap neutralized!")

Row count (331437) matches unique IDs (331437).

Missing values in features:
impressions_past_30d          0
clicks_past_30d               0
avg_position_past_30d    154699
dtype: int64

Fact 3: THE TRAP
If we train using 'LEAKED_future_data', the model will cheat with 100% accuracy.
Trap neutralized!


## 4. Data limits

Limitations of this dataset:

No intent/market context: The data indicates that impressions dropped, but cannot tell us why (e.g., search volume for the topic dropped globally).

Anonymized Queries: Google Search Console hides data for low-volume queries. We may miss long-tail keyword decay.

Directional, not causal: The model provides decision-support by flagging candidates likely to decay. It does not guarantee that refreshing the content will reverse it.

In [ ]:
# Data limits confirmed.

In [ ]:
import gc, json, os
import numpy as np, pandas as pd
from google.colab import userdata

HF = {"token": userdata.get("HF_TOKEN")}
BASE = "hf://datasets/FlyRank/internship-warehouse"
FEAT_MONTHS, LABEL_MONTH = ["2026-02", "2026-03", "2026-04"], "2026-05"
MIN_IMPR_WINDOW, MIN_CLICKS_APR, DROP_RATIO = 1000, 10, 0.80
KEYS = ["client_hash_id", "content_hash_id"]
COLS = ["report_date", *KEYS, "gsc_clicks", "gsc_impressions", "gsc_avg_position"]
assert LABEL_MONTH != "2026-06" and "2026-06" not in FEAT_MONTHS, "June is the sealed test month"

def month_agg(m):
    """One partition -> one row per (client, content). Loads only needed columns; frees RAM right after."""
    path = f"{BASE}/fact_content_daily_performance/month={m}/"
    try:
        d = pd.read_parquet(path, columns=COLS, storage_options=HF)
    except Exception:
        from huggingface_hub import HfFileSystem
        import pyarrow.parquet as pq
        fs = HfFileSystem(token=HF["token"])
        f = fs.ls(f"datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}", detail=False)[0]
        print("Actual schema:", pq.read_schema(f, filesystem=fs).names); raise
    dates = pd.to_datetime(d["report_date"])
    win = (dates.min().date(), dates.max().date(), len(d))
    impr = d["gsc_impressions"].fillna(0)
    pos = d["gsc_avg_position"].where(d["gsc_avg_position"] > 0)
    d = d.assign(clicks=d["gsc_clicks"].fillna(0), impr=impr,
                 pos_x_impr=(pos * impr).fillna(0), impr_pos=impr.where(pos.notna(), 0),
                 active=(impr > 0).astype("int16"))
    g = d.groupby(KEYS, observed=True).agg(clicks=("clicks", "sum"), impr=("impr", "sum"),
                                           pos_x_impr=("pos_x_impr", "sum"), impr_pos=("impr_pos", "sum"),
                                           active_days=("active", "sum")).reset_index()
    del d; gc.collect()
    return g, win

frames, windows = {}, {}
for m in FEAT_MONTHS + [LABEL_MONTH]:
    frames[m], windows[m] = month_agg(m)
    assert str(windows[m][0]).startswith(m) and str(windows[m][1]).startswith(m), f"{m} partition has dates outside its month"
    print(f"{m}: daily rows={windows[m][2]:,} | {windows[m][0]} -> {windows[m][1]} | contents={len(frames[m]):,}")

f = None
for m in FEAT_MONTHS:
    part = frames[m].rename(columns={c: f"{c}_{m}" for c in ["clicks", "impr", "pos_x_impr", "impr_pos", "active_days"]})
    f = part if f is None else f.merge(part, on=KEYS, how="outer")
num = [c for c in f.columns if c not in KEYS]
f[num] = f[num].fillna(0)
S = lambda p: sum(f[f"{p}_{m}"] for m in FEAT_MONTHS)
f["impressions_total"], f["clicks_total"] = S("impr"), S("clicks")
f["impressions_apr"], f["clicks_apr"], f["clicks_feb"] = f["impr_2026-04"], f["clicks_2026-04"], f["clicks_2026-02"]
f["momentum_apr_vs_feb"] = np.log1p(f["clicks_apr"]) - np.log1p(f["clicks_feb"])
f["ctr"] = np.where(f["impressions_total"] > 0, f["clicks_total"] / f["impressions_total"], np.nan)
f["weighted_position"] = np.where(S("impr_pos") > 0, S("pos_x_impr") / S("impr_pos").replace(0, np.nan), np.nan)
f["active_days"] = S("active_days")
FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
            "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]
f = f[KEYS + FEATURES]


may = frames[LABEL_MONTH][KEYS + ["clicks"]].rename(columns={"clicks": "clicks_may"})
n_before_eligibility = len(f)
f = f[(f["impressions_total"] >= MIN_IMPR_WINDOW) & (f["clicks_apr"] >= MIN_CLICKS_APR)]
f = f.merge(may, on=KEYS, how="left")
missing_in_may = int(f["clicks_may"].isna().sum())
f["clicks_may"] = f["clicks_may"].fillna(0)
f["y_decline_may"] = (f["clicks_may"] < DROP_RATIO * f["clicks_apr"]).astype(int)
LABEL_INGREDIENTS = ["clicks_may"]
frame = f.reset_index(drop=True)

assert not frame.duplicated(KEYS).any(), "grain broken: (client, content) must be unique"
assert not set(LABEL_INGREDIENTS) & set(FEATURES), "label ingredient leaked into features"
assert set(KEYS).isdisjoint(FEATURES), "IDs are grouping keys, never features"
share = frame["client_hash_id"].value_counts(normalize=True)
base_rate = float(frame["y_decline_may"].mean())
print(f"\nBefore eligibility: {n_before_eligibility:,} | Eligible rows: {len(frame):,} | Clients: {frame['client_hash_id'].nunique()}")
print(f"Base rate (decline in May): {base_rate:.3f} | Not seen in May (counted as decline): {missing_in_may:,}")
print(f"Largest client share of rows: {share.iloc[0]:.1%} | Top-5 clients: {share.head(5).sum():.1%}")
print("Per-client base rate spread (min / median / max, clients with >=30 rows):")
pc = frame.groupby("client_hash_id")["y_decline_may"].agg(["mean", "size"]); pc = pc[pc["size"] >= 30]["mean"]
print(f"  {pc.min():.2f} / {pc.median():.2f} / {pc.max():.2f}  over {len(pc)} clients")
print("Missing in features (share):"); print(frame[FEATURES].isna().mean().round(3).to_string())

os.makedirs("work/outputs", exist_ok=True)
frame.to_parquet("work/outputs/frame_dp_2026-05-01.parquet", index=False)
json.dump({"decision_point": "2026-05-01", "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH,
           "min_impr_window": MIN_IMPR_WINDOW, "min_clicks_apr": MIN_CLICKS_APR, "drop_ratio": DROP_RATIO,
           "rows_before_eligibility": int(n_before_eligibility), "rows": int(len(frame)),
           "clients": int(frame["client_hash_id"].nunique()), "base_rate": round(base_rate, 4),
           "not_seen_in_may": missing_in_may, "features": FEATURES, "label": "y_decline_may",
           "group_key": "client_hash_id", "sealed_month": "2026-06", "release": "v20260703"},
          open("work/outputs/data_contract_v2.json", "w"), indent=2)
print("\nSaved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)")

2026-02: daily rows=7,355,108 | 2026-02-01 -> 2026-02-28 | contents=321,546
2026-03: daily rows=9,841,378 | 2026-03-01 -> 2026-03-31 | contents=331,437
2026-04: daily rows=10,424,730 | 2026-04-01 -> 2026-04-30 | contents=362,172
2026-05: daily rows=11,687,376 | 2026-05-01 -> 2026-05-31 | contents=389,153

Before eligibility: 380,147 | Eligible rows: 16,513 | Clients: 36
Base rate (decline in May): 0.416 | Not seen in May (counted as decline): 0
Largest client share of rows: 26.1% | Top-5 clients: 71.9%
Per-client base rate spread (min / median / max, clients with >=30 rows):
  0.03 / 0.38 / 0.75  over 17 clients
Missing in features (share):
impressions_total      0.0
clicks_total           0.0
impressions_apr        0.0
clicks_apr             0.0
clicks_feb             0.0
momentum_apr_vs_feb    0.0
ctr                    0.0
weighted_position      0.0
active_days            0.0

Saved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.